# T3 — Mã hoá BGE-M3 trên Kaggle (GPU T4 ×2)

**Cài đặt notebook:** Accelerator = *GPU T4 ×2*, Internet = *On* (tải `BAAI/bge-m3` từ Hugging Face).

**Dữ liệu vào (Add Input → Datasets):**
- `mir-code`: thư mục dự án (`mir/`, `configs/`, `tests/`) — chỉ đọc, sẽ được chép sang `/kaggle/working/code`.
- `mir-data`: `sample/corpus_clean/`, `sample/chunks/` (đo tốc độ) và — khi chạy toàn bộ — `corpus_clean/`, `chunks/structure/`.

**Chia việc cho 3 người × 2 GPU = 6 shard:** người `m` (0, 1, 2) đặt `MEMBER = m`; GPU 0 chạy shard `2m/6`, GPU 1 chạy `2m+1/6`.
Mỗi khối 100.000 đoạn ghi ra `emb/structure/shard-XXXX/` và có `_DONE.json`; chạy lại notebook sẽ bỏ qua khối đã xong.

Thứ tự: (1) chuẩn bị → (2) đối chiếu với FlagEmbedding → (3) đo tốc độ → **dừng, báo số đo** → (4) chạy toàn bộ (mặc định tắt).

In [ ]:
# (1) Chuẩn bị
import os, shutil, subprocess, sys, json, glob
CODE_IN = '/kaggle/input/mir-code'          # sửa nếu tên dataset khác
DATA_IN = '/kaggle/input/mir-data'
WORK = '/kaggle/working'
CODE = f'{WORK}/code'
if not os.path.exists(CODE):
    shutil.copytree(CODE_IN, CODE)
os.chdir(CODE)
os.environ['MIR_DATA_DIR'] = DATA_IN
os.environ['MIR_WORK_DIR'] = WORK
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
subprocess.run('pip install -q pyyaml', shell=True)
print(subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run(f'df -h {WORK}', shell=True, capture_output=True, text=True).stdout)
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU', torch.cuda.device_count())

In [ ]:
# (2) Đối chiếu mir.encode với FlagEmbedding (thư viện chính thức) trên 32 đoạn của sample
# Cài FlagEmbedding vào thư mục riêng để không làm đổi phiên bản transformers của môi trường chính.
subprocess.run('pip install -q --no-deps --target /kaggle/tmp/flag FlagEmbedding', shell=True)
sys.path.insert(0, '/kaggle/tmp/flag')
import numpy as np, pyarrow.parquet as pq
from mir import encode
ch = pq.read_table(f'{DATA_IN}/sample/chunks/structure').to_pydict()
titles = encode.load_titles(f'{DATA_IN}/sample/corpus_clean')
texts = encode.make_inputs({'doc_id': ch['doc_id'][:32], 'text': ch['text'][:32]}, titles)
mine = encode.M3Encoder.load('BAAI/bge-m3', 'cuda:0', fp16=True, max_length=512)
d1, s1, _ = mine.encode(texts)
try:
    from FlagEmbedding import BGEM3FlagModel
    ref = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices=['cuda:0'])
    out = ref.encode(texts, max_length=512, return_dense=True, return_sparse=True)
    cos = (d1.astype(np.float32) * out['dense_vecs'].astype(np.float32)).sum(1)
    print('dense: cos(mir, FlagEmbedding) min =', cos.min().round(5))
    worst = 0.0; same_keys = 0
    for (ids, w), lw in zip(s1, out['lexical_weights']):
        ref_ = {int(k): float(v) for k, v in lw.items()}
        same_keys += set(ids.tolist()) == set(ref_)
        worst = max(worst, max((abs(ref_.get(int(t), 0) - float(x)) for t, x in zip(ids, w)), default=0))
    print(f'sparse: cùng tập token {same_keys}/{len(texts)}; chênh trọng số lớn nhất = {worst:.4f}')
    del ref
except Exception as e:
    print('Không chạy được FlagEmbedding để đối chiếu:', type(e).__name__, e)
del mine; torch.cuda.empty_cache()

In [ ]:
# (3) Đo tốc độ: 2 tiến trình chạy ĐỒNG THỜI trên 2 GPU, mỗi tiến trình 2.000 đoạn của sample
EST_CHUNKS = 8_600_000      # ước lượng số đoạn structure của toàn kho (results/t2_report.md)
cmd = ('python -m mir.encode --config configs/baseline.yaml --bench 2000 '
       f'--input {DATA_IN}/sample/chunks/structure --clean {DATA_IN}/sample/corpus_clean --est-chunks {EST_CHUNKS}')
procs = [subprocess.Popen(f'{cmd} --device cuda:{g} > {WORK}/bench_gpu{g}.log 2>&1', shell=True) for g in range(torch.cuda.device_count())]
[p.wait() for p in procs]
res = []
for g in range(len(procs)):
    print(open(f'{WORK}/bench_gpu{g}.log').read()[-1500:])
    res.append(json.load(open(f'{WORK}/results/t3_bench_cuda{g}.json')))
rate = sum(r['chunks_per_s'] for r in res)
print(f'Tổng tốc độ {len(res)} GPU: {rate:.0f} đoạn/s → toàn kho {EST_CHUNKS:,} đoạn ≈ {EST_CHUNKS / rate / 3600:.1f} giờ trên 1 máy; '
      f'≈ {EST_CHUNKS / rate / 3600 / 3:.1f} giờ/máy nếu 3 người chia đều; dung lượng ≈ {res[0]["est_storage_gb"]} GB')
# → DỪNG Ở ĐÂY, gửi số đo cho người duyệt trước khi chạy toàn bộ.

In [ ]:
# (4) Chạy toàn bộ — CHỈ bật sau khi đã duyệt số đo ở (3)
RUN_FULL = False
MEMBER = 0                   # 0, 1 hoặc 2
if RUN_FULL:
    cmd = ('python -m mir.encode --config configs/baseline.yaml '
           f'--input {DATA_IN}/chunks/structure --clean {DATA_IN}/corpus_clean --out {WORK}/emb/structure')
    procs = [subprocess.Popen(f'{cmd} --shard {2 * MEMBER + g}/6 --device cuda:{g} > {WORK}/encode_gpu{g}.log 2>&1', shell=True)
             for g in range(2)]
    [p.wait() for p in procs]
    for g in range(2):
        print(open(f'{WORK}/encode_gpu{g}.log').read()[-2000:])
    print(subprocess.run(f'du -sh {WORK}/emb; df -h {WORK}', shell=True, capture_output=True, text=True).stdout)